# Activity 2.8 – Colab ↔ VS Code bridge (A01286739)

**Question.** In a small synthetic sample of Pokémon-style trainer sessions, which creature *type* produces the highest average catch rate, and does session length relate to the number of catches?

**Data provenance.** All data are **synthetic**: they are generated inside this notebook with NumPy using a fixed random seed (`SEED = 2006`). No personal data, Google Drive files, local paths, APIs, or downloads are used, so any runtime (Colab, VS Code, local Jupyter) produces the same table.

**Expected output.**
1. A setup cell printing the Python, NumPy and pandas versions and the runtime that executed the notebook.
2. A 200-row synthetic table.
3. A summary table of catch rate by type and a correlation between session minutes and catches.
4. A short conclusion stating what the synthetic result does and does not show.

**Runtime vs. notebook.** This `.ipynb` file stores code, Markdown and saved outputs. It does **not** store the runtime (Python process, installed packages, variables in memory). Re-run all cells from a fresh runtime to reproduce the results.

## 1. Setup and dependencies
Only `numpy` and `pandas` are required; both are preinstalled in Google Colab. The cell installs them only if they are missing, so the dependency is recorded in the notebook itself.

In [ ]:
import sys, platform, importlib, subprocess

REQUIRED = ["numpy", "pandas"]
for pkg in REQUIRED:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
print("Python :", sys.version.split()[0])
print("NumPy  :", np.__version__)
print("pandas :", pd.__version__)
print("Platform:", platform.platform())
print("Runtime:", "Google Colab (hosted)" if IN_COLAB else "Local / other Jupyter kernel")

## 2. Synthetic data
Each row is one fictitious trainer session. Values come from fixed distributions with a seed, so the table is identical on every clean run.

In [ ]:
SEED = 2006
rng = np.random.default_rng(SEED)
N = 200

types = ["Fire", "Water", "Grass", "Electric", "Psychic"]
base_rate = {"Fire": 0.30, "Water": 0.40, "Grass": 0.45, "Electric": 0.25, "Psychic": 0.20}

session_type = rng.choice(types, size=N)
minutes = rng.integers(10, 91, size=N)
attempts = np.maximum(1, (minutes / 3 + rng.normal(0, 3, N)).round().astype(int))
p = np.array([base_rate[t] for t in session_type])
catches = rng.binomial(attempts, p)

df = pd.DataFrame({
    "session_id": np.arange(1, N + 1),
    "creature_type": session_type,
    "session_minutes": minutes,
    "attempts": attempts,
    "catches": catches,
})
df["catch_rate"] = (df["catches"] / df["attempts"]).round(3)

print(df.shape)
df.head()

## 3. Analysis

In [ ]:
summary = (
    df.groupby("creature_type")
      .agg(sessions=("session_id", "count"),
           attempts=("attempts", "sum"),
           catches=("catches", "sum"),
           mean_catch_rate=("catch_rate", "mean"))
      .assign(pooled_catch_rate=lambda t: (t["catches"] / t["attempts"]).round(3))
      .sort_values("pooled_catch_rate", ascending=False)
      .round(3)
)
corr = df["session_minutes"].corr(df["catches"])

assert len(df) == N and df["catch_rate"].between(0, 1).all()
print(f"Correlation session_minutes vs catches: {corr:.3f}")
summary

## 4. Conclusion

In [ ]:
best = summary.index[0]
print(f"Highest pooled catch rate: {best} ({summary.loc[best, 'pooled_catch_rate']:.1%}).")
print(f"Longer sessions go with more catches (r = {corr:.2f}).")
print("These results only reflect the synthetic generator (seed 2006); they are not evidence about real Pokémon GO players.")

**Interpretation.** The ranking reproduces the catch probabilities hard-coded in the generator (Grass and Water highest, Psychic lowest), and the positive correlation exists because attempts were generated in proportion to session minutes. The notebook demonstrates a reproducible workflow, not a real-world finding.

**Synchronization note.** Source of truth: GitHub (`notebooks/03_colab_bridge.ipynb`). The notebook was opened from GitHub in Google Colab and run with *Runtime → Run all* from a fresh session; the setup cell output records which runtime executed it. Saved outputs in this file are a snapshot — another person must re-run the notebook to get a live runtime.